# Medical MNIST Classification with PyTorch

Classifying 6 types of medical images using a CNN built from scratch.

**Dataset:** [Medical MNIST](https://www.kaggle.com/datasets/andrewmvd/medical-mnist)  
**Classes:** AbdomenCT, BreastMRI, ChestCT, CXR, Hand, HeadCT  
**Images:** 64x64 grayscale, 58,954 total  

## Table of Contents
1. Setup & Imports
2. Exploratory Data Analysis (EDA)
3. Dataset & DataLoader
4. CNN Model Definition
5. Training
6. Evaluation & Visualization

---
## 1. Setup & Imports

Load PyTorch and required libraries.  
Automatically uses GPU if available.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from collections import Counter

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms

from sklearn.metrics import confusion_matrix, classification_report
from tqdm import tqdm

# Check GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# Fix random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

---
## 2. Exploratory Data Analysis (EDA)

Before building a model, let's understand the data:
- Folder structure (one folder per class)
- Number of images per class (check for imbalance)
- Sample images

In [ ]:
import os

# Auto-detect dataset path (works with both Add Input and Import Notebook)
POSSIBLE_PATHS = [
    '/kaggle/input/medical-mnist',
    '/kaggle/input/datasets/andrewmvd/medical-mnist',
    '/kaggle/input/andrewmvd/medical-mnist',
]

DATA_DIR = None
for p in POSSIBLE_PATHS:
    if os.path.exists(p):
        DATA_DIR = p
        break

# If none found, download via kagglehub
if DATA_DIR is None:
    import subprocess
    subprocess.run(['pip', 'install', '-q', 'kagglehub'], check=True)
    import kagglehub
    DATA_DIR = kagglehub.dataset_download('andrewmvd/medical-mnist')

print(f'Using DATA_DIR: {DATA_DIR}')

# List class folders
classes = sorted([d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d))])
print(f'Classes ({len(classes)}): {classes}')

# Count images per class
class_counts = {}
for cls in classes:
    cls_path = os.path.join(DATA_DIR, cls)
    count = len(os.listdir(cls_path))
    class_counts[cls] = count
    print(f'  {cls}: {count} images')

print(f'\nTotal images: {sum(class_counts.values())}')

### Class Distribution

Check whether image counts are balanced across classes.  
Severe imbalance can bias the model toward majority classes.

In [ ]:
plt.figure(figsize=(10, 5))
bars = plt.bar(class_counts.keys(), class_counts.values(), color='steelblue')
plt.title('Class Distribution', fontsize=14)
plt.xlabel('Class')
plt.ylabel('Number of Images')
plt.xticks(rotation=45)

# Display count above each bar
for bar, count in zip(bars, class_counts.values()):
    plt.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 100,
             str(count), ha='center', va='bottom', fontsize=11)

plt.tight_layout()
plt.show()

### Sample Images

Display 2 images from each class to see what the data looks like.

In [ ]:
fig, axes = plt.subplots(2, len(classes), figsize=(18, 6))
fig.suptitle('Sample Images per Class', fontsize=14)

for col, cls in enumerate(classes):
    cls_path = os.path.join(DATA_DIR, cls)
    img_files = os.listdir(cls_path)[:2]
    
    for row, img_file in enumerate(img_files):
        img = Image.open(os.path.join(cls_path, img_file))
        axes[row, col].imshow(img, cmap='gray')
        axes[row, col].axis('off')
        if row == 0:
            axes[row, col].set_title(cls, fontsize=11)

plt.tight_layout()
plt.show()

# Check image dimensions
sample_img = Image.open(os.path.join(DATA_DIR, classes[0], os.listdir(os.path.join(DATA_DIR, classes[0]))[0]))
print(f'Image size: {sample_img.size}, Mode: {sample_img.mode}')

---
## 3. Dataset & DataLoader

Core PyTorch data structures:

- **Dataset**: Defines how to fetch a single sample (image + label)
- **DataLoader**: Serves data in batches to the model

```
Raw Data → Dataset (individual access) → DataLoader (batched) → Model
```

### Data Split
- **Train (70%)**: Model learns from this
- **Validation (15%)**: Monitor performance during training (not used for learning)
- **Test (15%)**: Final performance evaluation (never seen during training)

In [ ]:
class MedicalMNISTDataset(Dataset):
    """Custom Dataset for Medical MNIST images."""
    
    def __init__(self, data_dir, transform=None):
        self.data_dir = data_dir
        self.transform = transform
        self.classes = sorted(os.listdir(data_dir))
        self.class_to_idx = {cls: idx for idx, cls in enumerate(self.classes)}
        
        # Collect all image paths and labels
        self.samples = []
        for cls in self.classes:
            cls_path = os.path.join(data_dir, cls)
            if not os.path.isdir(cls_path):
                continue
            for img_name in os.listdir(cls_path):
                img_path = os.path.join(cls_path, img_name)
                self.samples.append((img_path, self.class_to_idx[cls]))
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert('L')  # Grayscale
        if self.transform:
            image = self.transform(image)
        return image, label

### Transforms (Preprocessing)

Image transformations before feeding to the model:
- `ToTensor()`: PIL image → PyTorch tensor (0~255 → 0.0~1.0)
- `Normalize()`: Shift to mean=0.5, std=0.5 → value range becomes -1 to 1

Normalization helps the model train more stably.

In [ ]:
# Image preprocessing pipeline
transform = transforms.Compose([
    transforms.Resize((64, 64)),       # Ensure uniform size
    transforms.ToTensor(),              # Convert to tensor (0~1)
    transforms.Normalize([0.5], [0.5])  # Normalize (-1~1)
])

# Create full dataset
full_dataset = MedicalMNISTDataset(DATA_DIR, transform=transform)
print(f'Total dataset size: {len(full_dataset)}')
print(f'Class mapping: {full_dataset.class_to_idx}')

# Train / Validation / Test split (70% / 15% / 15%)
total = len(full_dataset)
train_size = int(0.7 * total)
val_size = int(0.15 * total)
test_size = total - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(
    full_dataset, [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(42)
)

print(f'Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}')

### DataLoader

DataLoader serves data in **batches** to the model.

- `batch_size=64`: Process 64 images at a time
- `shuffle=True`: Randomize order each epoch (prevents overfitting)
- `num_workers=2`: Parallel data loading (faster)

In [ ]:
BATCH_SIZE = 64

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# Inspect one batch
images, labels = next(iter(train_loader))
print(f'Batch image shape: {images.shape}')  # [64, 1, 64, 64] = [batch, channels, height, width]
print(f'Batch label shape: {labels.shape}')  # [64]

---
## 4. CNN Model Definition

CNN (Convolutional Neural Network) is specialized for image recognition.

### Architecture
```
Input (1x64x64)
  ↓
Conv2d → ReLU → MaxPool  (Feature extraction stage 1)
  ↓
Conv2d → ReLU → MaxPool  (Feature extraction stage 2)
  ↓
Conv2d → ReLU → MaxPool  (Feature extraction stage 3)
  ↓
Flatten → FC → ReLU → Dropout → FC  (Classification)
  ↓
Output (6 class probabilities)
```

- **Conv2d**: Detects patterns (edges, textures) in images
- **ReLU**: Activation function — zeroes out negatives, adds non-linearity
- **MaxPool**: Halves spatial dimensions — reduces computation + adds translation invariance
- **Dropout**: Randomly disables neurons during training — prevents overfitting
- **FC (Fully Connected)**: Final classification based on extracted features

In [ ]:
class MedicalCNN(nn.Module):
    def __init__(self, num_classes=6):
        super(MedicalCNN, self).__init__()
        
        # Feature extractor (Convolutional Layers)
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),   # 1ch → 32ch
            nn.ReLU(),
            nn.MaxPool2d(2, 2),  # 64x64 → 32x32
            
            nn.Conv2d(32, 64, kernel_size=3, padding=1),  # 32ch → 64ch
            nn.ReLU(),
            nn.MaxPool2d(2, 2),  # 32x32 → 16x16
            
            nn.Conv2d(64, 128, kernel_size=3, padding=1), # 64ch → 128ch
            nn.ReLU(),
            nn.MaxPool2d(2, 2),  # 16x16 → 8x8
        )
        
        # Classifier (Fully Connected Layers)
        self.classifier = nn.Sequential(
            nn.Flatten(),                # 128*8*8 = 8192 → 1D vector
            nn.Linear(128 * 8 * 8, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)  # → 6 classes
        )
    
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

# Create model & move to GPU
model = MedicalCNN(num_classes=len(classes)).to(device)
print(model)

# Parameter count
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\nTotal params: {total_params:,} | Trainable: {trainable_params:,}')

---
## 5. Training

### Three essentials for training
1. **Loss Function**: Measures how far predictions are from ground truth
   - `CrossEntropyLoss`: Standard for multi-class classification
2. **Optimizer**: Updates model parameters to reduce loss
   - `Adam`: Most widely used optimizer
3. **Epochs**: Number of passes through the entire dataset

### Training Loop
```
Each epoch:
  1. Train: batch → predict → compute loss → backprop → update params
  2. Validate: evaluate performance without learning (monitor overfitting)
```

In [ ]:
# Hyperparameters
NUM_EPOCHS = 10
LEARNING_RATE = 0.001

# Loss function & optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# Training history
history = {
    'train_loss': [], 'train_acc': [],
    'val_loss': [], 'val_acc': []
}

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    """Run one epoch of training."""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()              # 1) Clear previous gradients
        outputs = model(images)            # 2) Forward pass
        loss = criterion(outputs, labels)  # 3) Compute loss
        loss.backward()                    # 4) Backpropagation
        optimizer.step()                   # 5) Update parameters
        
        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    
    return running_loss / total, correct / total


def evaluate(model, loader, criterion, device):
    """Evaluate model without training (for validation/test)."""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    
    return running_loss / total, correct / total

### Run Training

Train for 10 epochs, printing train/val loss and accuracy each epoch.

In [ ]:
print('Training started!')
print('=' * 70)

best_val_acc = 0.0

for epoch in range(NUM_EPOCHS):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)
    
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    
    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_model.pth')
    
    print(f'Epoch [{epoch+1:2d}/{NUM_EPOCHS}] '
          f'Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | '
          f'Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}'
          f'{" ★ Best" if val_acc >= best_val_acc else ""}')

print('=' * 70)
print(f'Training complete! Best Validation Accuracy: {best_val_acc:.4f}')

### Training Curves

Visualize how loss and accuracy change across epochs.

- **Both train & val improving together**: Normal learning
- **Train improves but val stalls/worsens**: Overfitting

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

epochs_range = range(1, NUM_EPOCHS + 1)

# Loss 그래프
ax1.plot(epochs_range, history['train_loss'], 'b-o', label='Train Loss')
ax1.plot(epochs_range, history['val_loss'], 'r-o', label='Val Loss')
ax1.set_title('Loss per Epoch')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Accuracy 그래프
ax2.plot(epochs_range, history['train_acc'], 'b-o', label='Train Acc')
ax2.plot(epochs_range, history['val_acc'], 'r-o', label='Val Acc')
ax2.set_title('Accuracy per Epoch')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 6. Evaluation & Visualization

Evaluate the trained model on **test data** — data it has never seen during training.  
This gives the true measure of model performance.

In [ ]:
# Load best model
model.load_state_dict(torch.load('best_model.pth'))

# Final evaluation on test data
test_loss, test_acc = evaluate(model, test_loader, criterion, device)
print(f'\n===== Test Results =====')
print(f'Test Loss: {test_loss:.4f}')
print(f'Test Accuracy: {test_acc:.4f} ({test_acc*100:.1f}%)')

### Confusion Matrix

Shows which classes get confused with each other.
- Diagonal: Correctly classified
- Off-diagonal: Misclassified (reveals which classes the model confuses)

In [ ]:
# 모든 예측값과 정답 수집
all_preds = []
all_labels = []

model.eval()
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        _, predicted = outputs.max(1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())

# Confusion Matrix 계산 & 시각화
cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=classes, yticklabels=classes)
plt.title('Confusion Matrix', fontsize=14)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

# 클래스별 상세 리포트
print('\n===== Classification Report =====')
print(classification_report(all_labels, all_preds, target_names=classes))

### Sample Predictions

Random test images with model predictions.
- **Green**: Correct
- **Red**: Wrong

In [ ]:
# 테스트 세트에서 랜덤 16장 시각화
fig, axes = plt.subplots(4, 4, figsize=(12, 12))

# 랜덤 인덱스
indices = np.random.choice(len(test_dataset), 16, replace=False)

model.eval()
with torch.no_grad():
    for i, idx in enumerate(indices):
        image, label = test_dataset[idx]
        
        # 모델 예측
        output = model(image.unsqueeze(0).to(device))
        _, predicted = output.max(1)
        pred_idx = predicted.item()
        
        # 시각화
        ax = axes[i // 4, i % 4]
        # 정규화 역변환 (시각화용)
        img_display = image.squeeze().numpy() * 0.5 + 0.5
        ax.imshow(img_display, cmap='gray')
        
        actual = classes[label]
        pred = classes[pred_idx]
        color = 'green' if label == pred_idx else 'red'
        ax.set_title(f'Pred: {pred}\nActual: {actual}', color=color, fontsize=9)
        ax.axis('off')

plt.suptitle('Sample Predictions (Green=Correct, Red=Wrong)', fontsize=14)
plt.tight_layout()
plt.show()

---
## Summary

1. **EDA**: 6 medical image classes, ~59K images, 64x64 grayscale
2. **Data Pipeline**: Custom Dataset, Train/Val/Test split, DataLoader
3. **Model**: 3-layer CNN (Conv→ReLU→Pool) + FC classifier
4. **Training**: 10 epochs, Adam optimizer, CrossEntropyLoss
5. **Evaluation**: Test accuracy, Confusion Matrix, Sample predictions

### Next Steps
- Transfer Learning (pretrained models: ResNet, EfficientNet)
- Data Augmentation (rotation, flipping for data diversity)
- More challenging datasets (Breast Cancer Histopathology, etc.)